# A Fairness + Interpretability Audit of a CNN

**Predicting `Male` from CelebA faces — does the model lean on hairstyle?**

This notebook runs the same two-instrument audit as before, on a **different** task, chosen
to test a different, contested claim: does a CNN predicting **gender from a face photo**
end up relying heavily on **hairstyle** — a social cue, not a biological one — rather than
facial structure?

We train an ordinary classifier to answer *is this person labeled "Male"?* on CelebA. In the
data, hairstyle attributes are strongly gendered — 94% of blond people are labeled female, and
a `Bald`/`Receding_Hairline` face is overwhelmingly labeled male — because hairstyle in a
photo dataset reflects social convention, not chromosomes. A model that finds hair a cheap,
reliable cue has every incentive to lean on it.

We then run **two** interpretability methods pointed at the same prediction — saliency and
Grad-CAM — and ask where the evidence lands: on the face, or on the hair? And we don't stop
at pictures: section 8 **physically hides the hair** in every test image and re-measures
accuracy, turning "the model looks at hair too much" into a number.

Alongside that, we audit the model's **fairness** across two attributes it was never trained
to know about: **hair color** (`Blond_Hair`) and **age** (`Young`). If hairstyle is doing the
classifying, whoever doesn't fit the stereotype for their labeled gender — a blond man, an
older woman — is who the model should fail hardest.

> **On the research.** Whether hairstyle drives gender/face-classification accuracy gaps is
> genuinely **contested** in the literature, not settled. Muthukumar et al. (2019) cropped
> hair out of commercial gender-classifier inputs and found the accuracy gap *persisted* —
> in their systems, makeup and facial structure were the bigger drivers, not hair length.
> Separately, Albiero et al. (2020) found that gendered hairstyles reduce the *visible face
> area* in recognition (matching) systems, which does hurt accuracy. Neither paper is about
> *our* model. That is exactly why we built the ablation in section 8: to test it directly,
> here, rather than assume the answer.
>
> **On the data.** CelebA's `Male` attribute is a binary, appearance-based label; real gender
> is neither binary nor reducible to appearance. We use it because it is the label the dataset
> provides and the one a real deployed classifier would be trained on — auditing *that* label
> is the point. CelebA's attribute annotations are also known to be noisy.

## Learning objectives

- Build a group-balanced test set that makes per-group error rates directly comparable.
- Audit a classifier with Fairlearn's `MetricFrame`, disaggregating accuracy, recall, and
  selection rate by a sensitive attribute the model never saw.
- Interpret equalized-odds and demographic-parity differences.
- Combine fairness metrics with saliency and Grad-CAM to move from *that* a model fails a group
  to *why*.
- Design an ablation that converts a visual hypothesis into a measured accuracy drop.
- State honestly what a single-model audit can and cannot conclude.

## Background

Everything mechanical here has already appeared elsewhere in the course. The saliency and
Grad-CAM implementations are lifted from `U2-2_CNN-8_Explainability.ipynb`, and the frozen
MobileNetV2 + linear head recipe comes from `U2-2_CNN-6_TransferLearning.ipynb`. The Fairlearn
tooling — `MetricFrame`, `equalized_odds_difference`, `demographic_parity_difference` — was
introduced on tabular data in `U1_Diabetes-7_Interp.ipynb`.

One idea is worth restating because the whole notebook turns on it. **Disaggregation** means
computing a metric separately within each group rather than once over the pool. Overall accuracy
is an average weighted by group size, so a model can fail a small group completely and barely
move the headline number. Two summary statistics recur below:

- **Equalized-odds difference** — the largest gap between groups in true-positive *or*
  false-positive rate. Zero means every group experiences the same error rates.
- **Demographic-parity difference** — the largest gap between groups in how often the model
  predicts the positive class, regardless of whether it is right.

They measure different things, and a model can satisfy either while violating the other.

## This notebook covers

1. The labels, and the correlations already present in CelebA
2. Building the training and balanced-test splits
3. Viewing the eight (gender × hair × age) cells
4. Training the classifier
5. Overall performance — the number that would go on a slide
6. The fairness audit, by hair color and by age
7. Two interpretability methods pointed at the same predictions
8. The ablation: hiding the hair and re-measuring
9. Review

**Prerequisites:** `U2-2_CNN-8_Explainability.ipynb` for saliency and Grad-CAM;
`U2-2_CNN-6_TransferLearning.ipynb` for the frozen-backbone recipe; `U1_Diabetes-7_Interp.ipynb`
for Fairlearn.

**Dataset:** CelebA, in the Kaggle `jessicali9530/celeba-dataset` layout — download it and point
`DATA_DIR` at your local copy. The notebook needs `img_align_celeba/`, `list_attr_celeba.csv`,
and `list_eval_partition.csv`.

**References:** listed in full in section 9.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

In [ ]:
import os
import tensorflow as tf

np.random.seed(0)
tf.random.set_seed(0)

# ── CONFIGURATION ──────────────────────────────────────────────────────────────
# Point DATA_DIR at your local CelebA (the Kaggle "jessicali9530/celeba-dataset" layout).
# Images are double-nested: <DATA_DIR>/img_align_celeba/img_align_celeba/000001.jpg
DATA_DIR   = r"C:\Users\Graham West\Python Notebooks\Meharry Teaching\Datasets\Celebrities"
IMG_DIR    = os.path.join(DATA_DIR, "img_align_celeba", "img_align_celeba")
ATTR_CSV   = os.path.join(DATA_DIR, "list_attr_celeba.csv")
PART_CSV   = os.path.join(DATA_DIR, "list_eval_partition.csv")

TARGET_ATTR = "Male"          # what the model predicts (gender, as CelebA labels it)
HAIR_ATTR   = "Blond_Hair"    # fairness-audit attribute #1
AGE_ATTR    = "Young"         # fairness-audit attribute #2

IMG_SIZE     = 224            # MobileNetV2's native size -> a 7x7 grid for Grad-CAM
N_PER_GENDER = 2500           # TRAIN images per gender, drawn WITHOUT reweighting hair/age
K_CELL       = 50             # TEST images per (gender x hair x age) cell -> balanced eval

print("images dir:", IMG_DIR)
print("exists:", os.path.isdir(IMG_DIR))

## 1. The labels — and the correlations already sitting in the data

Three binary attributes (±1 in the CSV, remapped to 0/1): the target (`Male`) and the two
audit attributes (`Blond_Hair`, `Young`). We are not going to manufacture any imbalance for
this notebook — unlike the earlier blond-hair notebook, where we deliberately preserved an
extreme correlation, here we just sample **normally** and see what a model trained on the
ordinary data picks up. If it leans on hair, that is CelebA's own bias speaking, not ours.

In [ ]:
attr = pd.read_csv(ATTR_CSV)
part = pd.read_csv(PART_CSV)

df = attr[["image_id", TARGET_ATTR, HAIR_ATTR, AGE_ATTR]].merge(part, on="image_id")

df["male"]  = (df[TARGET_ATTR] == 1).astype(int)
df["blond"] = (df[HAIR_ATTR]   == 1).astype(int)
df["young"] = (df[AGE_ATTR]    == 1).astype(int)
df["split"] = df["partition"].map({0: "train", 1: "val", 2: "test"})

train_pool = df[df.split == "train"]
test_pool  = df[df.split == "test"]

print(f"Overall base rates (train): male={train_pool.male.mean():.3f}  "
      f"blond={train_pool.blond.mean():.3f}  young={train_pool.young.mean():.3f}\n")

ct_hair = pd.crosstab(train_pool.male, train_pool.blond)
ct_hair.index = ["female", "male"]; ct_hair.columns = ["non-blond", "blond"]
print(f"TRAIN: gender x hair color\n{ct_hair}\n")

ct_age = pd.crosstab(train_pool.male, train_pool.young)
ct_age.index = ["female", "male"]; ct_age.columns = ["older", "young"]
print(f"TRAIN: gender x age\n{ct_age}")

print(f"\nAmong BLOND people: {(train_pool[train_pool.blond==1].male==0).mean()*100:.1f}% "
      f"are labeled female -- hair color is a strong, ready-made shortcut for 'female'.")
print(f"Among people labeled 'older' (Young=0): "
      f"{(train_pool[train_pool.young==0].male==1).mean()*100:.1f}% are labeled male -- "
      f"age is a weaker but real shortcut too.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ct_hair.plot.bar(ax=axes[0], color=["#4c72b0", "#dd8452"])
axes[0].set_title("Hair color, by labeled gender"); axes[0].set_ylabel("training images")
axes[0].tick_params(axis='x', rotation=0)
ct_age.plot.bar(ax=axes[1], color=["#4c72b0", "#dd8452"])
axes[1].set_title("Age, by labeled gender"); axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 2. Build the audit splits

- **Train**: `N_PER_GENDER` images per gender, sampled **at random** from the natural pool —
  no reweighting. Whatever hair/age correlation exists, exists because that is what CelebA
  looks like.
- **Test**: balanced across **all eight** combinations of (gender × hair × age) — 50 images
  per cell. This one stratified set drives *both* fairness audits (by hair, by age) from
  identical machinery, and even lets us peek at the intersection of the two.

The asymmetry is deliberate and it is the standard protocol. The **training** set should look
like the world the model would really be trained on, bias and all — that is the thing under
audit. The **test** set is the instrument, so it gets balanced: with 50 images in every cell, a
per-group recall is estimated on the same number of examples everywhere and the groups are
directly comparable. Balancing the training set instead would fix the bias rather than measure
it.

In [ ]:
def sample_ids(pool, n, rng):
    return pool.sample(n=min(n, len(pool)), random_state=rng).image_id.values

rng = 0

# --- TRAIN: N_PER_GENDER per gender, natural hair/age mix within each ---
train_ids = np.concatenate([
    sample_ids(train_pool[train_pool.male == m], N_PER_GENDER, rng) for m in (0, 1)
])
np.random.default_rng(rng).shuffle(train_ids)

# --- TEST: balanced K_CELL per (male x blond x young) cell -- 8 cells, 400 images ---
cells8 = [(m, b, y) for m in (0, 1) for b in (0, 1) for y in (0, 1)]
test_ids = np.concatenate([
    sample_ids(test_pool[(test_pool.male == m) & (test_pool.blond == b) & (test_pool.young == y)],
              K_CELL, rng + 1)
    for (m, b, y) in cells8
])

meta = df.set_index("image_id")
meta["sex"]  = np.where(meta.male == 1, "male", "female")
meta["hair"] = np.where(meta.blond == 1, "blond", "non-blond")
meta["age"]  = np.where(meta.young == 1, "young", "older")
meta["group8"] = meta.sex + " | " + meta.hair + " | " + meta.age

print(f"TRAIN: {len(train_ids)} images  ({meta.loc[train_ids].sex.value_counts().to_dict()})")
print(f"TEST : {len(test_ids)} images across {len(cells8)} balanced cells "
      f"({K_CELL} each)")
print(f"\nExample TEST cell counts:\n"
      f"{meta.loc[test_ids].group8.value_counts().sort_index().to_string()}")

In [ ]:
from PIL import Image

def load_images(ids):
    X = np.empty((len(ids), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    for k, img_id in enumerate(ids):
        im = Image.open(os.path.join(IMG_DIR, img_id)).convert("RGB").resize(
            (IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        X[k] = np.asarray(im, dtype=np.float32)
    return X

print("loading train images ...");  X_train = load_images(train_ids)
print("loading test images  ...");  X_test  = load_images(test_ids)

y_train = meta.loc[train_ids, "male"].values.astype(np.float32)
y_test  = meta.loc[test_ids,  "male"].values.astype(np.float32)

hair_test  = meta.loc[test_ids, "hair"].values     # sensitive feature #1 (Fairlearn)
age_test   = meta.loc[test_ids, "age"].values      # sensitive feature #2 (Fairlearn)
group_test = meta.loc[test_ids, "group8"].values   # full 8-way label, for picking examples

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## 3. Meet the eight cells

Every combination of gender × hair color × age, six faces each. Two rows worth a second look:
**blond men** (top-right-ish) and **bald/short-haired-coded older women** are the
counter-stereotype cases the rest of the notebook keeps returning to.

In [ ]:
fig, axes = plt.subplots(8, 6, figsize=(13, 17))
for r, (m, b, y) in enumerate(cells8):
    label = f"{'male' if m else 'female'} | {'blond' if b else 'non-blond'} | {'young' if y else 'older'}"
    ids_r = meta.loc[test_ids][meta.loc[test_ids].group8 == label].index[:6]
    for c, img_id in enumerate(ids_r):
        axes[r, c].imshow(Image.open(os.path.join(IMG_DIR, img_id)))
        axes[r, c].axis("off")
    axes[r, 0].set_ylabel(label, rotation=0, ha="right", va="center", fontsize=8.5)
    axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
plt.tight_layout(); plt.show()

## 4. Train the classifier

The same recipe as `U2-2_CNN-8_Explainability`: a **frozen** MobileNetV2 turns each face into
a 7×7×1280 feature tensor, and a single linear layer reads it off the globally-average-pooled
(GAP) features. As before, the head outputs a raw **logit** (no sigmoid), so the
gradient-based explanations don't die in a saturated activation.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Rescaling, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.losses import BinaryCrossentropy

base_model = MobileNetV2(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

feat_in = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = Rescaling(1./127.5, offset=-1)(feat_in)          # = mobilenet_v2.preprocess_input
x = base_model(x, training=False)
feature_model = Model(feat_in, x, name="features")
pool_model    = Model(feat_in, GlobalAveragePooling2D()(feature_model.output), name="pooled")

print("extracting features (train) ...");  P_train = pool_model.predict(X_train, batch_size=32, verbose=1)
print("extracting features (test)  ...");  P_test  = pool_model.predict(X_test,  batch_size=32, verbose=1)
print("pooled feature shape:", P_train.shape)

In [ ]:
def train_head(P, y, epochs=40):
    pi  = Input(shape=(P.shape[1],))
    out = Dense(1, name="logit")(Dropout(0.3)(pi))       # linear -> logit
    clf = Model(pi, out, name="clf")
    clf.compile(optimizer="adam", loss=BinaryCrossentropy(from_logits=True), metrics=["accuracy"])
    clf.fit(P, y, epochs=epochs, batch_size=64, verbose=0)
    return clf

clf = train_head(P_train, y_train)

# Re-assemble raw-pixels -> logit and conv-maps -> logit, sharing the SAME trained head.
# A positive logit means "male", matching CelebA's Male=1 coding.
ci = Input(shape=feature_model.output_shape[1:])
head_model = Model(ci, clf(GlobalAveragePooling2D()(ci)), name="head")
full_model = Model(feat_in, head_model(feature_model(feat_in)), name="full")
print(f"trained: {clf.count_params():,} head parameters")

## 5. Overall performance

The number that would go on a slide.

Note it down, and note how little it tells you. Everything section 6 finds is already present in
these same predictions — it is invisible here only because a single average has nowhere to put it.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

logit_test = clf.predict(P_test, verbose=0)[:, 0]
pred_test  = (logit_test > 0).astype(int)

acc = accuracy_score(y_test, pred_test)
print(f"Overall accuracy on the balanced test set: {acc:.3f}")

fig, ax = plt.subplots(figsize=(4.2, 3.4))
sns.heatmap(confusion_matrix(y_test, pred_test), annot=True, fmt="d", vmin=0,
            cmap="nipy_spectral", xticklabels=["female", "male"],
            yticklabels=["female", "male"], ax=ax)
ax.set_title(f"Overall — acc {acc:.3f}"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

## 6. The fairness audit — two attributes the model never saw

`MetricFrame` is the workhorse: give it true labels, predictions, and a **sensitive feature**, and
it computes each metric separately within every group. Nothing about the model changes — these are
the same predictions from section 5, simply sliced.

Four metrics per group, each answering a different question:

- **accuracy** — how often the model is right in this group.
- **recall (TPR)** — of the people in this group truly labeled male, how many were caught.
- **false-positive rate** — of those truly labeled female, how many were wrongly called male.
- **selection rate** — how often the model predicts "male" here at all, right or wrong.

### 6.1 By hair color

If hairstyle is doing real work, the group most likely to suffer is the one where hair color
contradicts the stereotype: a **blond man**. Watch recall (true-positive rate) *within* each
sex separately for blond vs. non-blond people — not overall accuracy, which a large majority
group can dominate.

In [ ]:
from fairlearn.metrics import (MetricFrame, selection_rate,
                               true_positive_rate, false_positive_rate,
                               equalized_odds_difference, demographic_parity_difference)

def audit_by(sensitive, name):
    mf = MetricFrame(
        metrics={"accuracy": accuracy_score,
                 "recall (TPR)":         true_positive_rate,
                 "false-positive rate":  false_positive_rate,
                 "selection rate":       selection_rate},
        y_true=y_test, y_pred=pred_test, sensitive_features=sensitive)
    print(f"By {name}:\n")
    print(mf.by_group.to_string(float_format=lambda v: f"{v:.3f}"), "\n")
    eod = equalized_odds_difference(y_test, pred_test, sensitive_features=sensitive)
    dpd = demographic_parity_difference(y_test, pred_test, sensitive_features=sensitive)
    print(f"Equalized-odds difference    : {eod:.3f}   (0 = perfectly equal error rates)")
    print(f"Demographic-parity difference: {dpd:.3f}")
    return mf

mf_hair = audit_by(hair_test, "hair color")
mf_hair.by_group[["accuracy", "recall (TPR)"]].plot.bar(
    figsize=(6.5, 4), rot=0, color=["#4c72b0", "#dd8452"])
plt.ylim(0, 1); plt.title("Gender prediction, split by hair color"); plt.ylabel("score")
plt.legend(loc="lower left"); plt.tight_layout(); plt.show()

### 6.2 The sharpest cut: recall on MEN, blond vs. non-blond

Recall (of the people truly labeled male, how many did the model catch?) split further by
hair color. If the model uses "not blond" as part of its idea of "male," blond men are exactly
who it should miss.

In [ ]:
is_male   = y_test == 1
male_hair = hair_test[is_male]
male_hit  = pred_test[is_male] == 1

recall_by_hair = pd.Series(male_hit, index=male_hair).groupby(level=0).mean()
for h in ["non-blond", "blond"]:
    print(f"Recall on people truly labeled MALE, {h:10s}: {recall_by_hair.get(h, float('nan')):.3f}")
gap_hair = recall_by_hair.get("non-blond", np.nan) - recall_by_hair.get("blond", np.nan)
print(f"\n   gap (non-blond - blond): {gap_hair:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["non-blond men", "blond men"],
       [recall_by_hair.get("non-blond", 0), recall_by_hair.get("blond", 0)],
       color=["#4c72b0", "#dd8452"])
for i, h in enumerate(["non-blond", "blond"]):
    ax.text(i, recall_by_hair.get(h, 0), f"{recall_by_hair.get(h, 0):.2f}", ha="center", va="bottom")
ax.set_ylim(0, 1); ax.set_ylabel("fraction correctly called 'male'")
ax.set_title("Does hair color change how well\nmen are recognised as men?")
plt.tight_layout(); plt.show()

### 6.3 By age

A second, independent audit: does accuracy hold up for people labeled "older" (`Young=0`) the
same as for those labeled "young"? Age is not the notebook's headline story, but it costs us
nothing to check, using the exact same balanced test set.

In [ ]:
mf_age = audit_by(age_test, "age")
mf_age.by_group[["accuracy", "recall (TPR)"]].plot.bar(
    figsize=(6.5, 4), rot=0, color=["#4c72b0", "#dd8452"])
plt.ylim(0, 1); plt.title("Gender prediction, split by age"); plt.ylabel("score")
plt.legend(loc="lower left"); plt.tight_layout(); plt.show()

The confusion matrix and overall accuracy in section 5 cannot see either of these gaps — they
only exist once you break the same predictions apart by a group the model was never trained
on. That is the entire argument for auditing by group instead of trusting one number.

## 7. Two ways to ask "what is it looking at?"

Saliency and Grad-CAM, both from `U2-2_CNN-8_Explainability`. Both are pointed at the same
thing: **evidence for whatever the model actually predicted** — so for a face called "male,"
positive heat means "pushed the model toward male."

Section 6 established *that* certain groups fare worse. These methods are how we start asking
*why*, and the two are chosen to be complementary: saliency is pixel-level and precise but noisy,
Grad-CAM is a coarse 7×7 blob but semantically reliable. Agreement between them is worth more
than either alone.

In [ ]:
def _signed_logit(img):
    lg = float(full_model.predict(img[None].astype("float32"), verbose=0)[0, 0])
    return lg, (1.0 if lg > 0 else -1.0)


def saliency(img, smooth=32, noise=0.10):
    """SmoothGrad (Smilkov et al., 2017): average |d logit / d pixel| over noisy copies.
    One backward pass per copy; averaging cancels the speckle a single gradient sample has."""
    _, sign = _signed_logit(img)
    x = tf.convert_to_tensor(img[None].astype("float32"))
    batch = x + tf.random.normal((smooth,) + img.shape, stddev=noise * 255.0)
    with tf.GradientTape() as tape:
        tape.watch(batch)
        score = full_model(batch, training=False)[:, 0] * sign
    g = tf.reduce_mean(tf.abs(tape.gradient(score, batch)), axis=0)
    return tf.reduce_max(g, axis=-1).numpy()


def grad_cam(img):
    """Grad-CAM (Selvaraju et al., 2017): weight each of the 7x7x1280 conv channels by the
    gradient of the predicted logit with respect to that channel, averaged over space, then
    sum and keep the positive part -- evidence FOR the predicted class, localised to a
    coarse 7x7 grid before upsampling."""
    x = tf.convert_to_tensor(img[None].astype("float32"))
    with tf.GradientTape() as tape:
        conv = feature_model(x, training=False)
        tape.watch(conv)
        logit = head_model(conv, training=False)[:, 0]
        score = logit * (1.0 if float(logit[0]) > 0 else -1.0)
    grads = tape.gradient(score, conv)
    w   = tf.reduce_mean(grads, axis=(1, 2))
    cam_map = tf.nn.relu(tf.einsum("bhwc,bc->bhw", conv, w))
    return tf.image.resize(cam_map[..., None], (IMG_SIZE, IMG_SIZE))[0, ..., 0].numpy()


def norm(h, pct=99):
    h = np.maximum(h, 0); hi = np.percentile(h, pct)
    return np.clip(h / (hi + 1e-8), 0, 1)


def explain(idx, tag=""):
    img = X_test[idx]
    p   = float(tf.sigmoid(full_model.predict(img[None].astype("float32"), verbose=0)[0, 0]))
    called = "male" if p > 0.5 else "female"
    maps = [("saliency (SmoothGrad)", saliency(img)), ("Grad-CAM", grad_cam(img))]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
    axes[0].imshow(img.astype("uint8")); axes[0].axis("off")
    axes[0].set_title(f"{group_test[idx]}\ntrue: {'male' if y_test[idx] else 'female'}"
                      f"  ->  called: {called} ({p:.2f})", fontsize=8.5)
    for ax, (name, h) in zip(axes[1:], maps):
        ax.imshow(img.astype("uint8")); ax.imshow(norm(h), cmap="jet", alpha=0.55)
        ax.set_title(name, fontsize=9); ax.axis("off")
    if tag: fig.suptitle(tag, y=1.05, fontsize=11)
    plt.tight_layout(); plt.show()

### 7.1 Typical cases — where does the evidence sit?

Start with ordinary, correctly-classified men and women. Watch whether the heat concentrates
on the **hair region** (top band of the frame) as much as, or more than, the face.

In [ ]:
correct_idx = np.where(pred_test == y_test)[0]
typical_m = [i for i in correct_idx if y_test[i] == 1][:4]
typical_f = [i for i in correct_idx if y_test[i] == 0][:4]

for i in typical_m:
    explain(i, "typical MALE prediction")
for i in typical_f:
    explain(i, "typical FEMALE prediction")

### 7.2 A systematic look: blond images, both sexes

Two anecdotes are not evidence. The fairness audit in section 6 flagged hair color
specifically — so before drawing any conclusion, look at a proper sample: several blond men
and several blond women side by side, with any **misclassifications** surfaced first since
those are the most informative cases. Does the model treat hair the same way in both groups,
or does it read "blond" as evidence for "female" regardless of who is actually wearing it?

In [ ]:
def sort_wrong_first(idx):
    """Put misclassified examples first -- they are the most instructive to look at."""
    wrong = idx[pred_test[idx] != y_test[idx]]
    right = idx[pred_test[idx] == y_test[idx]]
    return np.concatenate([wrong, right])

is_blond = hair_test == "blond"
blond_men_idx   = sort_wrong_first(np.where(is_blond & (y_test == 1))[0])
blond_women_idx = sort_wrong_first(np.where(is_blond & (y_test == 0))[0])

print(f"blond men in test:   {len(blond_men_idx):3d}  "
      f"({(pred_test[blond_men_idx] != y_test[blond_men_idx]).sum()} misclassified)")
print(f"blond women in test: {len(blond_women_idx):3d}  "
      f"({(pred_test[blond_women_idx] != y_test[blond_women_idx]).sum()} misclassified)")

N_SHOW = 8
for i in blond_men_idx[:N_SHOW]:
    wrong = pred_test[i] != y_test[i]
    explain(i, "blond MALE" + ("  --  MISCLASSIFIED" if wrong else "  --  correctly classified"))
for i in blond_women_idx[:N_SHOW]:
    wrong = pred_test[i] != y_test[i]
    explain(i, "blond FEMALE" + ("  --  MISCLASSIFIED" if wrong else "  --  correctly classified"))

### 7.3 Counter-stereotype cases — where the hair cue should break

Three more targeted cases, using CelebA's other hairstyle attributes the same way section 7.2
used blond hair:

- a man with **wavy hair or bangs** (`Male=1`, styled hair) — if the model leans on hairstyle,
  this is a plausible miss;
- a woman who is **bald or has a receding hairline** (`Male=0`, no visible "long hair") — same
  logic, the opposite direction;
- a **bald man** (`Male=1`, `Bald=1`) — there is no hair at all to fixate on, so this is the
  cleanest test of whether the model can fall back on facial structure alone.

In [ ]:
def pick(mask, fallback_mask, n=2):
    ids = np.where(mask)[0]
    if len(ids) == 0:
        ids = np.where(fallback_mask)[0]
    return list(ids[:n])

attr_test = attr.set_index("image_id").loc[test_ids]
wavy_or_bangs = ((attr_test["Wavy_Hair"] == 1) | (attr_test["Bangs"] == 1)).values
bald_or_recede = ((attr_test["Bald"] == 1) | (attr_test["Receding_Hairline"] == 1)).values
bald_only      = (attr_test["Bald"] == 1).values

styled_men   = pick((y_test == 1) & wavy_or_bangs,  y_test == 1)
hairless_wmn = pick((y_test == 0) & bald_or_recede, y_test == 0)
bald_men     = pick((y_test == 1) & bald_only,      y_test == 1)

for i in styled_men:
    explain(i, "man with styled/wavy hair or bangs -- counter-stereotype cue")
for i in hairless_wmn:
    explain(i, "woman who is bald or has a receding hairline -- counter-stereotype cue")
for i in bald_men:
    explain(i, "bald man -- no hair at all to lean on")

### 7.4 Reading the maps

For the **typical** cases (7.1), expect the heat to spread across a broad region that includes
a good deal of hair, not just eyes/nose/mouth — a first sign the model treats hairstyle as
load-bearing evidence, not incidental background.

For the **blond sample** (7.2), compare the two groups directly: if blond women's maps and
blond men's maps both light up the hair similarly, hair is being read as "blond," not as
"female" — a much weaker claim. If instead the misclassified blond men show *heavier* hair
attention than the blond women, and the flipped prediction, that is the shortcut caught in
the act: the model read the hair color as evidence of the label it usually comes with.

For the **counter-stereotype** cases (7.3), watch what happens when hair and face cues
disagree more broadly. If the maps keep lighting up a hairstyle that contradicts the label,
and the prediction flips to match the *hair* rather than the face, that is the same mechanism
again, from a different angle.

For the **bald man**, there is no hairstyle cue available at all — whatever the model does
here is necessarily face-based. Compare his confidence and his heatmap's concentration on
facial structure to the typical and blond-sample cases above.

Everything in this section is still *suggestive*, not conclusive. Heatmaps invite
over-reading, and a region can look hot without being decisive. That is exactly why section 8
stops looking and starts measuring.

## 8. Putting a number on it — how much do we lose without hair?

Pictures can be argued with. Here is a direct test: **grey out the top 40% of every test
image** — the band where hair almost always sits in an aligned CelebA face — and re-run the
exact same trained model. If hairstyle is genuinely load-bearing, accuracy should drop hard.
If the model is mostly reading the face, hiding the hair should barely matter.

This is the same logic behind Zeiler & Fergus's occlusion sensitivity (2014) — hide part of
the image, watch what the prediction loses — just applied once, at the scale of the whole
test set, as a blunt but conclusive experiment rather than a per-pixel sliding search.

> **One honest caveat about the ablation.** Greying a fixed top band removes forehead and some
> background along with the hair, and it hands the network a large flat grey rectangle unlike
> anything in its training data. Some of the accuracy drop is therefore attributable to
> distribution shift rather than to the missing hair specifically. The **comparison between
> groups** in the second cell is the more trustworthy signal: every group suffers the same
> distribution shift, so a group that loses substantially more than the others is losing
> something the others were not relying on.

In [ ]:
HAIR_BAND = 0.40                      # top fraction of the frame we grey out
cut = int(IMG_SIZE * HAIR_BAND)

X_test_nohair = X_test.copy()
X_test_nohair[:, :cut, :, :] = 127.5

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.6))
axes[0].imshow(X_test[0].astype("uint8"));        axes[0].set_title("original"); axes[0].axis("off")
axes[1].imshow(X_test_nohair[0].astype("uint8")); axes[1].set_title("hair band hidden"); axes[1].axis("off")
plt.tight_layout(); plt.show()

logit_nohair = full_model.predict(X_test_nohair, batch_size=64, verbose=1)[:, 0]
pred_nohair  = (logit_nohair > 0).astype(int)

acc_full   = accuracy_score(y_test, pred_test)
acc_nohair = accuracy_score(y_test, pred_nohair)
print(f"\nAccuracy WITH hair visible : {acc_full:.3f}")
print(f"Accuracy WITH hair hidden  : {acc_nohair:.3f}")
print(f"Drop                       : {acc_full - acc_nohair:.3f}")

In [ ]:
# Does hiding the hair hurt MORE for the counter-stereotype groups (blond men, older women)
# than for everyone else? If hair is a genuine shortcut, the group that most needed it should
# lose the most when it disappears.
for label, mask in [("blond men",        (y_test == 1) & (hair_test == "blond")),
                    ("non-blond men",   (y_test == 1) & (hair_test == "non-blond")),
                    ("women",           y_test == 0),
                    ("everyone",        np.ones_like(y_test, dtype=bool))]:
    a_full = accuracy_score(y_test[mask], pred_test[mask])
    a_noh  = accuracy_score(y_test[mask], pred_nohair[mask])
    print(f"{label:16s}  with-hair={a_full:.3f}   hair-hidden={a_noh:.3f}   drop={a_full-a_noh:+.3f}")

plt.figure(figsize=(5, 4))
plt.bar(["hair visible", "hair hidden"], [acc_full, acc_nohair], color=["#4c72b0", "#dd8452"])
for i, v in enumerate([acc_full, acc_nohair]):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.ylim(0, 1); plt.ylabel("accuracy"); plt.title("Gender accuracy, with vs. without visible hair")
plt.tight_layout(); plt.show()

## 9. Review

| Instrument | Section | Question it answers | What it cannot tell you |
|---|---|---|---|
| Overall accuracy | 5 | Is the model good on average? | Anything about who it fails |
| `MetricFrame` disaggregation | 6 | *Which groups* fare worse? | Why |
| Saliency + Grad-CAM | 7 | *Where* did the evidence sit? | Whether that region was decisive |
| Occlusion ablation | 8 | *How much* did that region matter? | Whether the cause is hair or distribution shift alone |

**The finding.** The confusion matrix in section 5 cannot see any of this — average accuracy
looked fine. Breaking the same predictions apart by hair color and age (section 6), looking at
a proper sample of blond men and women rather than one anecdote each (section 7.2), and then
literally hiding the hair (section 8), is what turns "maybe it uses hair" into a measured
result. Read your own run's accuracy-drop and recall-gap before trusting either story; this
notebook is built so both can be checked rather than assumed.

**Why the two audits belong together.** A fairness metric tells you *that* a model treats a
group worse; an interpretability method tells you *where* it looked, and the hair-occlusion
ablation in section 8 tells you *how much* that looking actually mattered — three levels of
the same question, and no one of them alone is convincing. A heatmap without the group
breakdown is a pretty picture; a group breakdown without the ablation is a correlation
someone could argue away.

**Why this matters beyond the number.** If a gender classifier partly keys on hairstyle, the
people it will fail are exactly the people whose hairstyle does not match the stereotype
associated with their labeled gender — a man with long hair, a woman with a shaved head, and
by the same logic, gender-nonconforming and transgender people whose presentation does not
map onto the training data's stereotypes at all. That is a real deployment harm, not an
academic curiosity, and it is invisible to a single accuracy number.

**Caveats to teach alongside the result:**
- CelebA's `Male` attribute is binary and appearance-based; it is not a measurement of gender
  identity. We audit the dataset's own encoded bias, not the concept of gender itself.
- The hair-reliance question is **contested in the literature** (see the intro) — this
  notebook's ablation tells you about *this* model, on *this* data, not gender classifiers in
  general.
- The section 8 ablation confounds "hair removed" with "large grey rectangle added"; the
  between-group comparison is the sounder reading.
- CelebA attribute labels are noisy, and `K_CELL=50` per group makes these estimates noisier
  still than a full-CelebA run; the *direction* of any gap is the reproducible part. If a gap
  looks small, that is itself informative — it means this particular frozen-backbone model
  did not take the shortcut as hard as expected, which is worth reporting honestly rather than
  the effect being assumed a priori.

**Where to take it further:** the mitigation tools from `U1_Diabetes-7_Interp.ipynb` apply
unchanged to these predictions — `ThresholdOptimizer` can equalize error rates post hoc, and
reweighting or balancing the training set attacks the correlation at its source. Auditing after
mitigation, with this same balanced test set, is the natural follow-up.

**References**
- Muthukumar et al. (2019), *Understanding Unequal Gender Classification Accuracy from Face
  Images* — found cropping out hair did **not** close the accuracy gap in commercial APIs;
  implicated makeup and facial structure instead
- Albiero et al. (2020), *Is Face Recognition Sexist? No, Gendered Hairstyles and Biology
  Are* — gendered hairstyles reduce usable *visible face area* in recognition (matching), a
  related but distinct mechanism from a classifier keying on hair as a positive cue
- Selvaraju et al. (2017), *Grad-CAM*
- Simonyan, Vedaldi & Zisserman (2013), *Deep Inside Convolutional Networks* — saliency
- Smilkov et al. (2017), *SmoothGrad*
- Zeiler & Fergus (2014), *Visualizing and Understanding Convolutional Networks* — occlusion
  sensitivity, the idea behind section 8's ablation
- Sagawa et al. (2020), *Distributionally Robust Neural Networks* — the CelebA worst-group
  evaluation protocol this notebook's balanced test set follows
- Geirhos et al. (2020), *Shortcut Learning in Deep Neural Networks*